In [16]:
from langchain_community.vectorstores import Cassandra
from langchain_ollama import OllamaLLM, OllamaEmbeddings
from datasets import load_dataset
import cassio
from pypdf import PdfReader

In [ ]:
ASTRA_DB_APPLICATION_TOKEN ="xxxx-xxxxx-xxxxx"
ASTRA_DB_ID = "xxxx-xxxxxx-xxxxxx"

In [18]:
pdfreader = PdfReader("Machine Learning.pdf")

In [19]:
from typing_extensions import Concatenate
raw_text = ''
for i, page in enumerate(pdfreader.pages):
    content = page.extract_text()
    if content:
        raw_text += content

In [20]:
cassio.init(token=ASTRA_DB_APPLICATION_TOKEN, database_id=ASTRA_DB_ID)

In [31]:
llm = OllamaLLM(model='llama3')
embeddings = OllamaEmbeddings(model='bge-m3')

In [23]:
astra_vector_store = Cassandra(
    embedding=embeddings,
    table_name="qa_machine_learning",
    session=None,
    keyspace=None
)

In [24]:
from langchain_text_splitters import CharacterTextSplitter
text_splitter = CharacterTextSplitter(
    separator='\n',
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
)
texts = text_splitter.split_text(raw_text)

In [25]:
texts[:50]

['Machine L earning Year ning is a   \ndeeplear ning. ai pr oject. \n \n \n \n \n \n \n \n \n \n \n© 2018 Andrew Ng.  All Rights R eserv ed. \n \n  \nPage 2 Machine Learning Yearning-Draft Andrew Ng Deeplearning.AI \nTable of Contents \n \n1 Why Machine Learning Strategy  \n2 How to use this book to help your team  \n3 Prerequisites and Notation  \n4 Scale drives machine learning progress  \n5 Your development and test sets  \n6 Your dev and test sets should come from the same distribution  \n7 How large do the dev/test sets need to be?  \n8 Establish a single-number evaluation metric for your team to optimize  \n9 Optimizing and satisficing metrics  \n10 Having a dev set and metric speeds up iterations  \n11 When to change dev/test sets and metrics  \n12 Takeaways: Setting up development and test sets  \n13 Build your first system quickly, then iterate  \n14 Error analysis: Look at dev set examples to evaluate ideas  \n15 Evaluating multiple ideas in parallel during error analysis',
 

In [27]:
astra_vector_store.add_texts(texts[:50])
print("Inserted %i lines." % len(texts[:50]))
astra_vector_retriever = astra_vector_store.as_retriever()

Inserted 50 lines.


In [32]:
first_question = True
while True:
    if first_question:
        query_text = input("\nAsk your question (or type 'quit' to exit): ").strip()
    else:
        query_text = input("\nWhat's yout next question (or type 'quit' to exit): ").strip()

    if query_text.lower() == 'quit':
        break

    if query_text == '':
        continue

    first_question = False

    print("\nQuestion: \"%s\"" % query_text)

    # Retrieve relevant documents
    docs = astra_vector_retriever.invoke(query_text)

    # Combine retrieved documents into context
    context = "\n\n".join([doc.page_content for doc in docs])

    # Send context + question to LLM
    prompt = f"""Use the following context to answer the question.

    Context: {context}

    Question:{query_text}
    """

    answer = llm.invoke(prompt).strip()

    print("\nAnswer: \"%s\"" % answer)

    print("\nFirst Documents by relevance:")

    for doc, score in astra_vector_store.similarity_search_with_score(query_text, k=4):
        print("     [%0.4f] \"%s ...\"" % (score, doc.page_content[:84]))


Question: "what is linear regression"

Answer: "According to the context provided, linear regression is mentioned as one of the supervised learning algorithms, specifically mentioned under the section "If you have taken a Machine Learning course such as my machine learning MOOC on Coursera, or if you have experience applying supervised learning, you will be able to understand this text."

In more detail, it says: "Supervised learning algorithms include linear regression, logistic regression, and neural networks.""

First Documents by relevance:
     [0.7528] "your team with a few such changes, I hope that you can become the superhero of your  ..."
     [0.7528] "your team with a few such changes, I hope that you can become the superhero of your  ..."
     [0.7360] "Machine L earning Year ning is a   
deeplear ning. ai pr oject. 
 
 
 
 
 
 
 
 
 
  ..."
     [0.7360] "Machine L earning Year ning is a   
deeplear ning. ai pr oject. 
 
 
 
 
 
 
 
 
 
  ..."

Question: "what is machine 